# derived_8.4-hybrid-lstm-2.0 — Strict Static/Temporal Split (Static → XGBoost, Temporal → LSTM)

In `derived_8.4-hybrid-lstm-1.6`, the hybrid XGBoost received the 54-feature backbone (49 of them temporal/rolling) plus an LSTM context computed from largely the same temporal features; SHAP showed the context dominating (56–86% of SHAP share). This experiment removes that overlap with a **strict split**: **XGBoost gets only the 18 static (time-invariant) features** directly, and **the LSTM gets only the 79 temporal/rolling features** (its 45 temporal inputs from 1.6 + the 34 temporal backbone features 1.6 did not feed it), retrained at hidden sizes H ∈ {40, 20, 16, 8, 4}. [1.6] leaderboard rows are appended as references.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

# Handle both: run from notebooks/ dir and run from the experiment dir itself
_cwd = Path.cwd().resolve()
if (_cwd / "experiment" / "derived_8.4-hybrid-lstm-2.0").is_dir():
    NOTEBOOK_DIR = _cwd / "experiment" / "derived_8.4-hybrid-lstm-2.0"
else:
    NOTEBOOK_DIR = _cwd
PROJECT_ROOT = NOTEBOOK_DIR.parents[2]
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(NOTEBOOK_DIR))

from lstm.train import (
    train_lstm_for_hidden, TEMPORAL_FEATURES, STATIC_FEATURES, SEQ_LEN, HIDDEN_SIZES,
)
from eval_hybrid.data import load_hybrid_experiment_data, verify_static_temporal_split
from eval_hybrid.evaluator import HybridStrategyEvaluator
from eval_hybrid.shap_analysis import run_full_shap_analysis
from run_eval import (
    load_reference_1_6, build_variants, _model_name, _candidate_id,
)

with open(NOTEBOOK_DIR / "config.yaml") as f:
    config = yaml.safe_load(f)

ARTIFACTS_DIR = NOTEBOOK_DIR / "artifacts"
MODELS_DIR = NOTEBOOK_DIR / "models"
DATA_DIR = PROJECT_ROOT / config.get("data_dir", "data/splits/derived_8.4")
hidden_sizes = [int(h) for h in config["hidden_sizes"]]
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# --- Feature split audit (static -> XGBoost, temporal -> LSTM) ---
train_df_check = pd.read_csv(DATA_DIR / "train.csv")
split_evidence = verify_static_temporal_split(train_df_check, STATIC_FEATURES, TEMPORAL_FEATURES)

print("Setup complete. Hidden sizes:", hidden_sizes)
print(f"Split: {len(STATIC_FEATURES)} static -> XGBoost | {len(TEMPORAL_FEATURES)} temporal -> LSTM | disjoint")

[SplitCheck] static=18 (all constant per station), temporal=79, disjoint=True


Setup complete. Hidden sizes: [40, 20, 16, 8, 4]
Split: 18 static -> XGBoost | 79 temporal -> LSTM | disjoint


## Phase 1-3: V21 Training (temporal-only input) + Raw Representation Extraction

For each hidden size H in {40, 20, 16, 8, 4}: train the V21 BiLSTM+Attn on the 79 temporal features (1 seed), evaluate it standalone, and extract raw ctx (2H-dim) / head_hidden (H-dim) / head_pre_relu (H-dim) vectors into `artifacts/h{H}/`. If those files already exist, the training step is skipped.

In [2]:
lstm_metrics_map = {}
for h in hidden_sizes:
    h_dir = ARTIFACTS_DIR / f"h{h}"
    raw_names = ("ctx", "head_hidden", "head_pre_relu")
    has_raw = all((h_dir / f"{n}_{s}.npy").exists()
                  for n in raw_names for s in ("train", "val", "test"))
    has_metrics = (h_dir / "lstm_metrics.json").exists()
    if has_raw and has_metrics:
        print(f"[H{h}] Found existing representations + metrics. Skipping training.")
        with open(h_dir / "lstm_metrics.json") as f:
            lstm_metrics_map[h] = json.load(f)
        continue
    h_dir.mkdir(parents=True, exist_ok=True)
    best_seed_info, lstm_metrics = train_lstm_for_hidden(h, DATA_DIR, h_dir, MODELS_DIR)
    lstm_metrics_map[h] = lstm_metrics
    print(f"[H{h}] Best seed: {best_seed_info['seed']} (val_rmse={best_seed_info['val_rmse']:.5f})")

print("\nLSTM-only test metrics per hidden size:")
for h in hidden_sizes:
    t = lstm_metrics_map[h]["test"]
    print(f"  H{h}: R2={t['r2']:.4f} RMSE={t['rmse']:.5f} MAE={t['mae']:.5f}")

[H40] Found existing representations + metrics. Skipping training.
[H20] Found existing representations + metrics. Skipping training.


[H16] Found existing representations + metrics. Skipping training.


[H8] Found existing representations + metrics. Skipping training.
[H4] Found existing representations + metrics. Skipping training.

LSTM-only test metrics per hidden size:
  H40: R2=0.5336 RMSE=0.06957 MAE=0.05574
  H20: R2=0.6477 RMSE=0.06047 MAE=0.04773
  H16: R2=0.5254 RMSE=0.07018 MAE=0.05504
  H8: R2=0.5616 RMSE=0.06745 MAE=0.05424
  H4: R2=0.5838 RMSE=0.06572 MAE=0.05215


## Phase 4-5: XGBoost Static Baselines + Hybrid Models

Train the 2 static-only tabular baselines (Global Single, Clustering_V0_Full_k2 — no temporal cluster additions, per the strict split design) and, for every (hidden size, representation) combination, the hybrid `[18 static + raw repr]` models under both strategies. No PCA is applied anywhere.

In [3]:
static_feats = list(config["static_features"])

data_base = load_hybrid_experiment_data(PROJECT_ROOT, NOTEBOOK_DIR, config, repr_type="ctx", hidden_size=hidden_sizes[0])
eval_global = HybridStrategyEvaluator(data_base, config, "Global_Single", models_dir=MODELS_DIR)
eval_v0 = HybridStrategyEvaluator(data_base, config, "Clustering_V0_Full_k2", models_dir=MODELS_DIR)

results = {}
results[f"Global Single ({len(static_feats)} Static)"] = eval_global.fit_and_evaluate(
    model_name=f"Global Single ({len(static_feats)} Static)",
    candidate_id=f"Global_Single_{len(static_feats)}_Static",
    global_features=static_feats,
)
results[f"Clustering_V0_Full_k2 ({len(static_feats)} Static, c0=0, c1=0)"] = eval_v0.fit_and_evaluate(
    model_name=f"Clustering_V0_Full_k2 ({len(static_feats)} Static, c0=0, c1=0)",
    candidate_id=f"Clustering_V0_k2_{len(static_feats)}_Static",
    global_features=static_feats,
    cluster_additions={"0": [], "1": []},
)

for h, repr_type in build_variants(hidden_sizes):
    data_v = load_hybrid_experiment_data(PROJECT_ROOT, NOTEBOOK_DIR, config, repr_type=repr_type, hidden_size=h)
    ev_g = HybridStrategyEvaluator(data_v, config, "Global_Single", models_dir=MODELS_DIR)
    ev_c = HybridStrategyEvaluator(data_v, config, "Clustering_V0_Full_k2", models_dir=MODELS_DIR)
    results[_model_name(h, repr_type, "Global_Single")] = ev_g.fit_and_evaluate(
        model_name=_model_name(h, repr_type, "Global_Single"),
        candidate_id=_candidate_id(h, repr_type, "Global_Single"),
        global_features=data_v.hybrid_features,
    )
    results[_model_name(h, repr_type, "Clustering_V0_Full_k2")] = ev_c.fit_and_evaluate(
        model_name=_model_name(h, repr_type, "Clustering_V0_Full_k2"),
        candidate_id=_candidate_id(h, repr_type, "Clustering_V0_Full_k2"),
        global_features=data_v.hybrid_features,
        cluster_additions={"0": [], "1": []},
    )

print(f"Trained {len(results)} XGBoost models.")

/scratch/group/p.cis250607.000/MDR-Project/notebooks/.venv/lib64/python3.12/site-packages/xgboost/core.py:751: UserWarning: [01:50:15] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Trained 32 XGBoost models.


## Phase 6: SHAP Feature Importance Analysis

Accelerated SHAP using XGBoost's native C++/CUDA `pred_contribs` on the test set for every fitted model; produces `shap_importance_summary.csv`, `shap_top20_comparison.csv`, `shap_ctx_vs_tabular.json`, and the summary plot.

In [4]:
eval_map = {}
for h, repr_type in build_variants(hidden_sizes):
    dv = load_hybrid_experiment_data(PROJECT_ROOT, NOTEBOOK_DIR, config, repr_type=repr_type, hidden_size=h)
    ev_g = HybridStrategyEvaluator(dv, config, "Global_Single", models_dir=MODELS_DIR)
    ev_c = HybridStrategyEvaluator(dv, config, "Clustering_V0_Full_k2", models_dir=MODELS_DIR)
    eval_map[_model_name(h, repr_type, "Global_Single")] = ev_g
    eval_map[_model_name(h, repr_type, "Clustering_V0_Full_k2")] = ev_c

eval_map[f"Global Single ({len(static_feats)} Static)"] = eval_global
eval_map[f"Clustering_V0_Full_k2 ({len(static_feats)} Static, c0=0, c1=0)"] = eval_v0

shap_info = run_full_shap_analysis(eval_map, results, ARTIFACTS_DIR)
print(f"SHAP computed for {len(shap_info['shap_results'])} models.")

Executing Accelerated SHAP Feature Importance Analysis


[SHAP] Computing SHAP values for: Global Single (18 Static)...


  -> Done in 0.117s (pred_contribs)


[SHAP] Computing SHAP values for: Clustering_V0_Full_k2 (18 Static, c0=0, c1=0)...


  -> Done in 0.052s (pred_contribs)


...

[SHAP] Computing SHAP values for: Clustering_V0_Full_k2 (18 Static, c0=0, c1=0 + 4 Pre-ReLU [H4])...


  -> Done in 0.474s (pred_contribs)


[SHAP] Saved summary table to /scratch/group/p.cis250607.000/MDR-Project/notebooks/experiment/derived_8.4-hybrid-lstm-2.0/artifacts/shap_importance_summary.csv


[SHAP] Saved visualization plot to /scratch/group/p.cis250607.000/MDR-Project/notebooks/experiment/derived_8.4-hybrid-lstm-2.0/artifacts/shap_summary_plots.png


SHAP computed for 32 models.


## Results: Leaderboard

Combined leaderboard: this experiment's 2 static baselines + 30 hybrid models + 5 LSTM-only rows, sorted by pooled test R², with [1.6] reference rows appended for direct comparison.

In [5]:
records = [r.as_record() for r in results.values()]
for h in hidden_sizes:
    lstm_test = lstm_metrics_map[h]["test"]
    records.append({
        "model_name": f"BiLSTM+Attn H{h} (LSTM-only, temporal-only input)",
        "pooled_r2": lstm_test["r2"],
        "pooled_rmse": lstm_test["rmse"],
        "pooled_ubrmse": lstm_test["ubrmse"],
        "pooled_bias": lstm_test["bias"],
        "pooled_mae": lstm_test["mae"],
        "pooled_pearson": float("nan"),
        "year_2023_r2": float("nan"), "year_2024_r2": float("nan"), "year_2025_r2": float("nan"),
    })
df = pd.DataFrame(records)
df_ref = load_reference_1_6()
if not df_ref.empty:
    df = pd.concat([df, df_ref], ignore_index=True)
df = df.sort_values("pooled_r2", ascending=False).reset_index(drop=True)
display_cols = ["model_name", "pooled_r2", "pooled_rmse", "pooled_mae", "pooled_pearson"]
display(df[display_cols].round(4))

# Per-regime breakdown for the README (collected from the Clustering models).
per_regime_records = []
for name, res in results.items():
    if not getattr(res, "cluster_metrics", None):
        continue
    for cl, m in res.cluster_metrics.items():
        per_regime_records.append({
            "model_name": name, "cluster": cl,
            "n_train": m["n_train"], "n_test": m["n_test"],
            "r2": m["r2"], "rmse": m["rmse"], "ubrmse": m["ubrmse"],
            "bias": m["bias"], "mae": m["mae"], "pearson": m["pearson"],
        })
df_regime = pd.DataFrame(per_regime_records)
print(f"Per-regime rows: {len(df_regime)}")


[Ref] Loaded 37 reference rows from derived_8.4-hybrid-lstm-1.6.


,model_name,pooled_r2,pooled_rmse,pooled_mae,pooled_pearson
0,"[1.6] Clustering_V0_Full_k2 (54 Backbone, c0=0...",0.8150,0.0438,0.0337,0.9056
1,[1.6] Global Single (54 Backbone),0.7792,0.0479,0.0371,0.8894
2,"[1.6] Clustering_V0_Full_k2 (54 Backbone, c0=0...",0.7550,0.0504,0.0390,0.8843
3,"[1.6] Clustering_V0_Full_k2 (54 Backbone, c0=0...",0.7529,0.0506,0.0389,0.8827
4,[1.6] Global Single (54 Backbone + 40 CTX [H20]),0.7524,0.0507,0.0393,0.8858
...,...,...,...,...,...
69,"BiLSTM+Attn H40 (LSTM-only, temporal-only input)",0.5336,0.0696,0.0557,NaN
70,"[1.6] BiLSTM+Attn H4 (LSTM-only, V21)",0.5263,0.0701,0.0546,NaN
71,"BiLSTM+Attn H16 (LSTM-only, temporal-only input)",0.5254,0.0702,0.0550,NaN
72,Global Single (18 Static),0.0241,0.1006,0.0830,0.2091


Per-regime rows: 48


## Save Artifacts & Generate README

Persist the leaderboard CSV, per-regime CSV, metrics JSON, prediction arrays, and the SHAP artifacts, then regenerate `README.md` from the executed notebook outputs.

In [6]:
df.to_csv(ARTIFACTS_DIR / "summary_records.csv", index=False)
df_regime.to_csv(ARTIFACTS_DIR / "per_regime_records.csv", index=False)
with open(ARTIFACTS_DIR / "metrics.json", "w") as f:
    json.dump({k: r.as_record() for k, r in results.items()}, f, indent=2)

from run_eval import generate_readme
generate_readme(df, df_regime, shap_info)
print("README generated.")



[Generated] /scratch/group/p.cis250607.000/MDR-Project/notebooks/experiment/derived_8.4-hybrid-lstm-2.0/README.md


README generated.
